```
---
title: Black list of small molecules in BioPAX
tags: BioPAX, smallMolecules, SPARQL, Reactome, ChEBI, blacklist
lang: en
version: 0.3
date: 2026-07-01
---
```

In [1]:
#import importlib
import IPython
#import json
#import matplotlib.pyplot as plt
#import networkx as nx
#import os
import pandas
#import rdflib
#import rdflib.namespace
import sparqldataframe
from SPARQLWrapper import SPARQLWrapper, JSON
#import sys

In [2]:
reactomeVersion = 96 # 2026-03-25
#chebiVersion = 252 # CHECKED BY QUERY BELOW
#endpointURL = "http://localhost:3030/reactome/query"
#endpointURL = "http://localhost:3030/chebi/query"
endpointURL = "http://localhost:3030/reactomeChEBI/query"
rdfFormat = "turtle"

In [3]:
prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>

PREFIX CHEBI: <http://purl.obolibrary.org/obo/CHEBI_>
PREFIX chebirel: <http://purl.obolibrary.org/obo/CHEBI#>
PREFIX oboInOwl: <http://www.geneontology.org/formats/oboInOwl#>

PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>

PREFIX reactome: <http://www.reactome.org/biopax/{}/48887#>
""".format(reactomeVersion)

In [4]:
query="""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:<http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dc: <http://purl.org/dc/elements/1.1/>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?ontology ?versionIRI (REPLACE(REPLACE(STR(?versionIRI), 'http://purl.obolibrary.org/obo/chebi/', ''), '/chebi.owl', '') AS ?versionNumber)
WHERE {
  ?ontology rdf:type owl:Ontology .
  ?ontology owl:versionIRI ?versionIRI .
}
"""

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

chebiVersion = int(results["results"]["bindings"][0]["versionNumber"]["value"])
print("ChEBI version: {}".format(chebiVersion))

ChEBI version: 252


In [5]:
def displaySparqlResults(results):
    '''
    Displays as HTML the result of a SPARQLWrapper query in a Jupyter notebook.
    
        Parameters:
            results (dictionnary): the result of a call to SPARQLWrapper.query().convert()
    '''
    variableNames = results['head']['vars']
    tableCode = '<table><tr><th>{}</th></tr><tr>{}</tr></table>'.format('</th><th>'.join(variableNames), '</tr><tr>'.join('<td>{}</td>'.format('</td><td>'.join([row[vName]['value'] if vName in row.keys() else "&nbsp;" for vName in variableNames]))for row in results["results"]["bindings"]))
    IPython.display.display(IPython.display.HTML(tableCode))

# 1. Blacklist composition

In [6]:
moleculeBlackList = ["CHEBI:15377", "CHEBI:24636", "CHEBI:15378", "CHEBI:15379", "CHEBI:57783", "CHEBI:13392", "CHEBI:16474", "CHEBI:58349","CHEBI:18009", "CHEBI:13390", "CHEBI:25523", "CHEBI:25524", "CHEBI:30616", "CHEBI:15422", "CHEBI:16761", "CHEBI:456216", "CHEBI:57945", "CHEBI:16908","CHEBI:57540", "CHEBI:15846", "CHEBI:16526", "CHEBI:29888", "CHEBI:35782", "CHEBI:68836", "CHEBI:18361", "CHEBI:43474", "CHEBI:35780", "CHEBI:18367", "CHEBI:26078", "CHEBI:77740", "CHEBI:28931", "CHEBI:58307", "CHEBI:17877", "CHEBI:17552", "CHEBI:58189", "CHEBI:37565", "CHEBI:15996"]
moleculeBlackListStr = " ".join(moleculeBlackList)
moleculeBlackListComma = ", ".join(moleculeBlackList)

In [7]:
print("Nb molecules in blacklist: {}".format(len(moleculeBlackList)))

Nb molecules in blacklist: 37


# 2. Redundancy

## 2.1 Pairs of redundant molecules

In [8]:
query = """
SELECT DISTINCT ?blacklistMoleculeDescendant ?blacklistMoleculeDescendantLabel ?blacklistMoleculeAncestor ?blacklistMoleculeAncestorLabel
WHERE {{
  VALUES ?blacklistMoleculeDescendant {{ {} }}
  VALUES ?blacklistMoleculeAncestor {{ {} }}
  
  ?blacklistMoleculeDescendant rdfs:subClassOf+ ?blacklistMoleculeAncestor .
  ?blacklistMoleculeDescendant rdfs:label ?blacklistMoleculeDescendantLabel .
  ?blacklistMoleculeAncestor rdfs:label ?blacklistMoleculeAncestorLabel .
}}
ORDER BY ?blacklistMoleculeDescendant ?blacklistMoleculeAncestor
""".format(moleculeBlackListStr, moleculeBlackListStr)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb pairs: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb pairs: 17


blacklistMoleculeDescendant,blacklistMoleculeDescendantLabel,blacklistMoleculeAncestor,blacklistMoleculeAncestorLabel
http://purl.obolibrary.org/obo/CHEBI_13390,NAD(P)(+),http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)
http://purl.obolibrary.org/obo/CHEBI_13392,NAD(P)H,http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)
http://purl.obolibrary.org/obo/CHEBI_15846,NAD(+),http://purl.obolibrary.org/obo/CHEBI_13390,NAD(P)(+)
http://purl.obolibrary.org/obo/CHEBI_15846,NAD(+),http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH,http://purl.obolibrary.org/obo/CHEBI_13392,NAD(P)H
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH,http://purl.obolibrary.org/obo/CHEBI_25523,NADP
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH,http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)
http://purl.obolibrary.org/obo/CHEBI_16908,NADH,http://purl.obolibrary.org/obo/CHEBI_13392,NAD(P)H
http://purl.obolibrary.org/obo/CHEBI_16908,NADH,http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)
http://purl.obolibrary.org/obo/CHEBI_18009,NADP(+),http://purl.obolibrary.org/obo/CHEBI_13390,NAD(P)(+)


## 2.2 Non-redundant molecules (kernel)

In [9]:
query = """
SELECT DISTINCT ?blacklistMolecule ?blacklistMoleculeLabel
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  FILTER NOT EXISTS {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  }}
}}
ORDER BY ?blacklistMolecule
""".format(moleculeBlackListStr, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb non-redundant blacklist molecules: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb non-redundant blacklist molecules: 26


blacklistMolecule,blacklistMoleculeLabel
http://purl.obolibrary.org/obo/CHEBI_15377,water
http://purl.obolibrary.org/obo/CHEBI_15378,hydron
http://purl.obolibrary.org/obo/CHEBI_15379,dioxygen
http://purl.obolibrary.org/obo/CHEBI_15422,ATP
http://purl.obolibrary.org/obo/CHEBI_15996,GTP
http://purl.obolibrary.org/obo/CHEBI_16526,carbon dioxide
http://purl.obolibrary.org/obo/CHEBI_16761,ADP
http://purl.obolibrary.org/obo/CHEBI_17552,GDP
http://purl.obolibrary.org/obo/CHEBI_17877,FADH2
http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)


## 2.3 Redundant blacklist molecules

In [10]:
query = """
SELECT DISTINCT ?blacklistMolecule ?blacklistMoleculeLabel
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  #FILTER NOT EXISTS {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  #}}
}}
ORDER BY ?blacklistMolecule
""".format(moleculeBlackListStr, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb redundant blacklist molecules: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb redundant blacklist molecules: 11


blacklistMolecule,blacklistMoleculeLabel
http://purl.obolibrary.org/obo/CHEBI_13390,NAD(P)(+)
http://purl.obolibrary.org/obo/CHEBI_13392,NAD(P)H
http://purl.obolibrary.org/obo/CHEBI_15846,NAD(+)
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH
http://purl.obolibrary.org/obo/CHEBI_16908,NADH
http://purl.obolibrary.org/obo/CHEBI_18009,NADP(+)
http://purl.obolibrary.org/obo/CHEBI_18361,diphosphate(4-)
http://purl.obolibrary.org/obo/CHEBI_18367,phosphate(3-)
http://purl.obolibrary.org/obo/CHEBI_24636,proton
http://purl.obolibrary.org/obo/CHEBI_25523,NADP


In [10]:
query = """
SELECT DISTINCT ?blacklistMolecule ?blacklistMoleculeLabel ?isCoreBlacklist
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  OPTIONAL {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  }}
  BIND ((! BOUND(?blacklistMoleculeAncestor)) AS ?isCoreBlacklist)
}}
ORDER BY ?blacklistMolecule
""".format(moleculeBlackListStr, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb redundant blacklist molecules: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb redundant blacklist molecules: 37


blacklistMolecule,blacklistMoleculeLabel,isCoreBlacklist
http://purl.obolibrary.org/obo/CHEBI_13390,NAD(P)(+),false
http://purl.obolibrary.org/obo/CHEBI_13392,NAD(P)H,false
http://purl.obolibrary.org/obo/CHEBI_15377,water,true
http://purl.obolibrary.org/obo/CHEBI_15378,hydron,true
http://purl.obolibrary.org/obo/CHEBI_15379,dioxygen,true
http://purl.obolibrary.org/obo/CHEBI_15422,ATP,true
http://purl.obolibrary.org/obo/CHEBI_15846,NAD(+),false
http://purl.obolibrary.org/obo/CHEBI_15996,GTP,true
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH,false
http://purl.obolibrary.org/obo/CHEBI_16526,carbon dioxide,true


## 2.4 Descendants of non-redundant blacklist molecules

In [12]:
query = """
SELECT DISTINCT ?blacklistMoleculeDescendant ?blacklistMoleculeDescendantLabel ?isInBlacklist
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  FILTER NOT EXISTS {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  }}

  ?blacklistMoleculeDescendant rdfs:subClassOf+ ?blacklistMolecule .
  ?blacklistMoleculeDescendant rdfs:label ?blacklistMoleculeDescendantLabel .
  BIND (?blacklistMoleculeDescendant IN ({}) AS ?isInBlacklist)
}}
ORDER BY ?blacklistMoleculeDescendant
""".format(moleculeBlackListStr, moleculeBlackListComma, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb of descendants of non-redundant blacklist molecule: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb of descendants of non-redundant blacklist molecule: 41


blacklistMoleculeDescendant,blacklistMoleculeDescendantLabel,isInBlacklist
http://purl.obolibrary.org/obo/CHEBI_13389,NAD,false
http://purl.obolibrary.org/obo/CHEBI_13390,NAD(P)(+),true
http://purl.obolibrary.org/obo/CHEBI_13392,NAD(P)H,true
http://purl.obolibrary.org/obo/CHEBI_134125,(R)-NADHX,false
http://purl.obolibrary.org/obo/CHEBI_15644,(S)-NADHX(1+),false
http://purl.obolibrary.org/obo/CHEBI_15846,NAD(+),true
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH,true
http://purl.obolibrary.org/obo/CHEBI_16908,NADH,true
http://purl.obolibrary.org/obo/CHEBI_18009,NADP(+),true
http://purl.obolibrary.org/obo/CHEBI_18361,diphosphate(4-),true


## 2.5 Number of descendant by non-redundant blacklist molecule

In [14]:
query = """
SELECT DISTINCT ?blacklistMolecule ?blacklistMoleculeLabel ?blacklistMoleculeDescendant ?blacklistMoleculeDescendantLabel ?isInBlacklist
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  FILTER NOT EXISTS {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  }}

  OPTIONAL {{
    ?blacklistMoleculeDescendant rdfs:subClassOf+ ?blacklistMolecule .
    ?blacklistMoleculeDescendant rdfs:label ?blacklistMoleculeDescendantLabel .
    BIND (?blacklistMoleculeDescendant IN ({}) AS ?isInBlacklist)
  }}
}}
ORDER BY ?blacklistMolecule ?blacklistMoleculeDescendant
""".format(moleculeBlackListStr, moleculeBlackListComma, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb of descendants of non-redundant blacklist molecule: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb of descendants of non-redundant blacklist molecule: 60


blacklistMolecule,blacklistMoleculeLabel,blacklistMoleculeDescendant,blacklistMoleculeDescendantLabel,isInBlacklist
http://purl.obolibrary.org/obo/CHEBI_15377,water,http://purl.obolibrary.org/obo/CHEBI_29374,ditritium oxide,false
http://purl.obolibrary.org/obo/CHEBI_15377,water,http://purl.obolibrary.org/obo/CHEBI_29375,diprotium oxide,false
http://purl.obolibrary.org/obo/CHEBI_15377,water,http://purl.obolibrary.org/obo/CHEBI_33806,deuterium hydrogen oxide,false
http://purl.obolibrary.org/obo/CHEBI_15377,water,http://purl.obolibrary.org/obo/CHEBI_33811,hydrogen tritium oxide,false
http://purl.obolibrary.org/obo/CHEBI_15377,water,http://purl.obolibrary.org/obo/CHEBI_33813,((18)O)water,false
http://purl.obolibrary.org/obo/CHEBI_15377,water,http://purl.obolibrary.org/obo/CHEBI_41981,dideuterium oxide,false
http://purl.obolibrary.org/obo/CHEBI_15378,hydron,http://purl.obolibrary.org/obo/CHEBI_24636,proton,true
http://purl.obolibrary.org/obo/CHEBI_15378,hydron,http://purl.obolibrary.org/obo/CHEBI_29233,deuteron,false
http://purl.obolibrary.org/obo/CHEBI_15378,hydron,http://purl.obolibrary.org/obo/CHEBI_29234,triton,false
http://purl.obolibrary.org/obo/CHEBI_15379,dioxygen,http://purl.obolibrary.org/obo/CHEBI_26689,singlet dioxygen,false


## 2.6 Non-redundant blacklist molecule with >= 1 descendant not in blacklist

In [43]:
query = """
SELECT DISTINCT ?blacklistMolecule ?blacklistMoleculeLabel 
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  FILTER NOT EXISTS {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  }}

  ?blacklistMoleculeDescendant rdfs:subClassOf+ ?blacklistMolecule .
  ?blacklistMoleculeDescendant rdfs:label ?blacklistMoleculeDescendantLabel .
  FILTER (?blacklistMoleculeDescendant NOT IN ({}) )
}}
ORDER BY ?blacklistMolecule ?blacklistMoleculeDescendant
""".format(moleculeBlackListStr, moleculeBlackListComma, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb of non-redundant blacklist molecule with >=1 descendant not in blacklist: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb of non-redundant blacklist molecule with >=1 descendant not in blacklist: 7


blacklistMolecule,blacklistMoleculeLabel
http://purl.obolibrary.org/obo/CHEBI_15377,water
http://purl.obolibrary.org/obo/CHEBI_15378,hydron
http://purl.obolibrary.org/obo/CHEBI_15379,dioxygen
http://purl.obolibrary.org/obo/CHEBI_15996,GTP
http://purl.obolibrary.org/obo/CHEBI_25524,NAD(P)
http://purl.obolibrary.org/obo/CHEBI_35780,phosphate ion
http://purl.obolibrary.org/obo/CHEBI_35782,diphosphate ion


# 3. Participation in biochemical reactions (are blacklist molecules ubiquitous?)

In [18]:
query = """
SELECT (COUNT(DISTINCT ?reaction) AS ?nbBiochemicalReactions)
WHERE {
  ?reaction rdf:type bp3:BiochemicalReaction .
  #?reaction rdf:type/(rdfs:subClassOf*) bp3:BiochemicalReaction . # same result
}
"""

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb of biochemical reactions: {}".format(results["results"]["bindings"][0]["nbBiochemicalReactions"]["value"]))

Nb of biochemical reactions: 15994


In [11]:
query = """
SELECT ?blacklistMolecule ?blacklistMoleculeLabel ?chebiID (COUNT(DISTINCT ?reaction) AS ?nbBiochemicalReactions)
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  BIND(REPLACE(STR(?blacklistMolecule), "http://purl.obolibrary.org/obo/CHEBI_", "CHEBI:") AS ?chebiID) # ChEBI URL -> ident
  #BIND(IRI(REPLACE(?chebiID, "CHEBI:", "http://purl.obolibrary.org/obo/CHEBI_")) AS ?blacklistMolecule) # ChEBI ident -> URL

  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .

  OPTIONAL {{
  ?reaction rdf:type/(rdfs:subClassOf*) bp3:BiochemicalReaction .
  ?reaction (bp3:left|bp3:right)/bp3:entityReference/bp3:xref [
    rdf:type bp3:UnificationXref ;
    bp3:db "ChEBI" ;
    bp3:id ?chebiID
  ] .
  }}
}}
GROUP BY ?blacklistMolecule ?blacklistMoleculeLabel ?chebiID
ORDER BY DESC(?nbBiochemicalReactions)
""".format(moleculeBlackListStr)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

print("Nb molecules: {}".format(len(results["results"]["bindings"])))
displaySparqlResults(results)

Nb molecules: 37


blacklistMolecule,blacklistMoleculeLabel,chebiID,nbBiochemicalReactions
http://purl.obolibrary.org/obo/CHEBI_30616,ATP(4-),CHEBI:30616,1874
http://purl.obolibrary.org/obo/CHEBI_456216,ADP(3-),CHEBI:456216,1653
http://purl.obolibrary.org/obo/CHEBI_15377,water,CHEBI:15377,1451
http://purl.obolibrary.org/obo/CHEBI_15378,hydron,CHEBI:15378,1060
http://purl.obolibrary.org/obo/CHEBI_43474,hydrogenphosphate,CHEBI:43474,551
http://purl.obolibrary.org/obo/CHEBI_15379,dioxygen,CHEBI:15379,387
http://purl.obolibrary.org/obo/CHEBI_57783,NADPH(4-),CHEBI:57783,269
http://purl.obolibrary.org/obo/CHEBI_58349,NADP(3-),CHEBI:58349,245
http://purl.obolibrary.org/obo/CHEBI_37565,GTP(4-),CHEBI:37565,244
http://purl.obolibrary.org/obo/CHEBI_58189,GDP(3-),CHEBI:58189,222


<div class="alert alert-block alert-danger">
<b>BUG:</b> why aren't ADT and ATP consumed by any reaction?. Check <a href="https://reactome.org/PathwayBrowser/#/R-HSA-71850">R-HSA-71850</a> that consumes ADP and produces ATP
</div>

<div class="alert alert-block alert-success">
<b>Hint:</b> The blacklist references ADP as CHEBI:16761, but Reactome uses CHEBI:456216. Check whether (1) ChEBI uses several identifiers for the same molecule and (2) whether it provides mappings (e.g. rdfs:seeAlso) between them
</div>

In [12]:
query = """
SELECT DISTINCT ?molecule ?moleculeName ?moleculeChebiIdent ?moleculeChebiLabel
WHERE {
  VALUES ?reactionIdent { "R-HSA-71850" }
  ?reaction rdf:type ?reactionType .
  ?reaction bp3:xref [
    bp3:db "Reactome" ;
    bp3:id ?reactionIdent
  ] .

  ?reaction bp3:left ?molecule .
  ?molecule rdf:type ?moleculeType .
  ?molecule bp3:displayName ?moleculeName .
  OPTIONAL {
    ?molecule bp3:entityReference/bp3:xref [
    #?molecule bp3:xref [
      bp3:db "ChEBI" ;
      bp3:id ?moleculeChebiIdent
    ] .
  }
  BIND(IRI(REPLACE(?moleculeChebiIdent, "CHEBI:", "http://purl.obolibrary.org/obo/CHEBI_")) AS ?moleculeChebiURL) # ChEBI ident -> URL
  ?moleculeChebiURL rdfs:label ?moleculeChebiLabel .
}
"""

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

displaySparqlResults(results)

molecule,moleculeName,moleculeChebiIdent,moleculeChebiLabel
http://www.reactome.org/biopax/96/48887#SmallMolecule36,ADP,CHEBI:456216,ADP(3-)
http://www.reactome.org/biopax/96/48887#SmallMolecule2619,"1,3BPG",CHEBI:57604,3-phosphonato-D-glyceroyl phosphate(4-)


In [44]:
query = """
SELECT ?blacklistMolecule ?blacklistMoleculeLabel ?chebiID (COUNT(DISTINCT ?reaction) AS ?nbBiochemicalReactions)
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}
  BIND(REPLACE(STR(?blacklistMolecule), "http://purl.obolibrary.org/obo/CHEBI_", "CHEBI:") AS ?chebiID) # ChEBI URL -> ident
  #BIND(IRI(REPLACE(?chebiID, "CHEBI:", "http://purl.obolibrary.org/obo/CHEBI_")) AS ?blacklistMolecule) # ChEBI ident -> URL

  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .
  
  ?reaction rdf:type/(rdfs:subClassOf*) bp3:BiochemicalReaction .
  ?reaction (bp3:left|bp3:right)/(bp3:component*)/bp3:entityReference/bp3:xref [
    rdf:type bp3:UnificationXref ;
    bp3:db "ChEBI" ;
    bp3:id ?chebiID
  ] .
}}
GROUP BY ?blacklistMolecule ?blacklistMoleculeLabel ?chebiID
ORDER BY DESC(?nbBiochemicalReactions)
""".format(moleculeBlackListStr)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

#print("Nb of biochemical reactions: {}".format(results["results"]["bindings"][0]["nbBiochemicalReactions"]["value"]))
displaySparqlResults(results)

blacklistMolecule,blacklistMoleculeLabel,chebiID,nbBiochemicalReactions
http://purl.obolibrary.org/obo/CHEBI_30616,ATP(4-),CHEBI:30616,1965
http://purl.obolibrary.org/obo/CHEBI_456216,ADP(3-),CHEBI:456216,1696
http://purl.obolibrary.org/obo/CHEBI_15377,water,CHEBI:15377,1451
http://purl.obolibrary.org/obo/CHEBI_15378,hydron,CHEBI:15378,1060
http://purl.obolibrary.org/obo/CHEBI_37565,GTP(4-),CHEBI:37565,748
http://purl.obolibrary.org/obo/CHEBI_43474,hydrogenphosphate,CHEBI:43474,553
http://purl.obolibrary.org/obo/CHEBI_15379,dioxygen,CHEBI:15379,387
http://purl.obolibrary.org/obo/CHEBI_58189,GDP(3-),CHEBI:58189,365
http://purl.obolibrary.org/obo/CHEBI_57783,NADPH(4-),CHEBI:57783,269
http://purl.obolibrary.org/obo/CHEBI_58349,NADP(3-),CHEBI:58349,245


In [51]:
query = """
SELECT ?blacklistMoleculeDescendant ?blacklistMoleculeDescendantLabel ?chebiID ?isInBlacklist (COUNT(DISTINCT ?reaction) AS ?nbBiochemicalReactions)
WHERE {{
  VALUES ?blacklistMolecule {{ {} }}

  ?blacklistMolecule rdfs:label ?blacklistMoleculeLabel .
  
  FILTER NOT EXISTS {{
    ?blacklistMolecule rdfs:subClassOf+ ?blacklistMoleculeAncestor .
    FILTER (?blacklistMoleculeAncestor IN ({}))
  }}

  ?blacklistMoleculeDescendant rdfs:subClassOf+ ?blacklistMolecule .
  ?blacklistMoleculeDescendant rdfs:label ?blacklistMoleculeDescendantLabel .
  BIND (?blacklistMoleculeDescendant IN ({}) AS ?isInBlacklist)

  BIND(REPLACE(STR(?blacklistMoleculeDescendant), "http://purl.obolibrary.org/obo/CHEBI_", "CHEBI:") AS ?chebiID) # ChEBI URL -> ident
  #BIND(IRI(REPLACE(?chebiID, "CHEBI:", "http://purl.obolibrary.org/obo/CHEBI_")) AS ?blacklistMoleculeDescendant) # ChEBI ident -> URL
  ?reaction rdf:type/(rdfs:subClassOf*) bp3:BiochemicalReaction .
  #?reaction (bp3:left|bp3:right)/bp3:entityReference/bp3:xref [
  ?reaction (bp3:left|bp3:right)/(bp3:component*)/bp3:entityReference/bp3:xref [
    rdf:type bp3:UnificationXref ;
    bp3:db "ChEBI" ;
    bp3:id ?chebiID
  ] .
}}
GROUP BY ?blacklistMoleculeDescendant ?blacklistMoleculeDescendantLabel ?chebiID ?isInBlacklist
ORDER BY DESC(?nbBiochemicalReactions)
""".format(moleculeBlackListStr, moleculeBlackListComma, moleculeBlackListComma)

sparql = SPARQLWrapper(endpointURL)
sparql.setQuery(prefixes+query)
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

#print("Nb of biochemical reactions: {}".format(results["results"]["bindings"][0]["nbBiochemicalReactions"]["value"]))
displaySparqlResults(results)

blacklistMoleculeDescendant,blacklistMoleculeDescendantLabel,chebiID,isInBlacklist,nbBiochemicalReactions
http://purl.obolibrary.org/obo/CHEBI_43474,hydrogenphosphate,CHEBI:43474,true,553
http://purl.obolibrary.org/obo/CHEBI_33019,diphosphate(3-),CHEBI:33019,false,282
http://purl.obolibrary.org/obo/CHEBI_16474,NADPH,CHEBI:16474,true,61
http://purl.obolibrary.org/obo/CHEBI_18009,NADP(+),CHEBI:18009,true,56
http://purl.obolibrary.org/obo/CHEBI_18367,phosphate(3-),CHEBI:18367,true,4
